# Chapter 11 (Part 1): News Ingestion Pipeline
**Financial AI in Practice** | Manning Publications

This notebook covers:
- Building multi-source news data fetchers (NewsAPI, WNN RSS, Google RSS, GDELT)
- Persisting raw data to Parquet for reproducibility
- Cleaning and unifying heterogeneous news sources into a single golden dataset
- Preparing text for embedding with source-aware metadata

## 11.2 Implementing the News Ingestion and Processing Pipeline
### Listing 11.1 — NewsAPI fetcher with Parquet persistence

In [ ]:
from newsapi import NewsApiClient
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv, find_dotenv
import os

# Keys come from the repo-root .env (appendix A): copy .env.example to .env and fill it in
load_dotenv(find_dotenv(usecwd=True))
NEWS_API_KEY = os.getenv("NEWS_API_KEY")
if not NEWS_API_KEY:
    raise RuntimeError("NEWS_API_KEY is not set: copy .env.example to .env in the repo root and add your key.")

def fetch_newsapi(keywords, days_back=30, api_key=NEWS_API_KEY):
    newsapi = NewsApiClient(api_key=api_key)

    from_date = (datetime.now() - timedelta(days=days_back)).strftime('%Y-%m-%d')
    to_date = datetime.now().strftime('%Y-%m-%d')

    query = ' OR '.join([f'"{kw}"' if ' ' in kw else kw for kw in keywords])

    response = newsapi.get_everything(
        q=query,
        from_param=from_date,
        to=to_date,
        language='en',
        sort_by='publishedAt',
        page_size=100
    )  #A

    rows = []
    for article in response.get('articles', []):
        rows.append({
            'collection_date': datetime.now().strftime('%Y-%m-%d'),
            'title': article.get('title', ''),
            'description': article.get('description', ''),
            'url': article.get('url', ''),
            'source_name': article.get('source', {}).get('name', 'Unknown'),
            'published_at': article.get('publishedAt'),
            'published_dt': pd.to_datetime(article.get('publishedAt'))
        })

    df = pd.DataFrame(rows)
    df['data_source'] = 'NewsAPI'
    return df.drop_duplicates(subset=['url']).reset_index(drop=True)

# Execute and persist (collecting from August 2025 onward)
keywords = ['uranium', 'nuclear energy', 'Kazatomprom', 'Cameco']
df_newsapi = fetch_newsapi(keywords=keywords, days_back=30, api_key=NEWS_API_KEY)
df_newsapi.to_parquet('newsapi_recent.parquet')
print(f"Saved {len(df_newsapi)} NewsAPI articles")

#A This fetcher retrieves a single page of up to 100 articles. NewsAPI's get_everything is paginated; for full coverage across the date range, loop over pages until all results are retrieved.

### Listing 11.2 — World Nuclear News RSS fetcher with Parquet persistence

In [ ]:
import feedparser

def fetch_wnn_rss():
    feed = feedparser.parse('https://world-nuclear-news.org/rss')
    rows = []
    for entry in feed.entries:
        rows.append({
            'title': entry.get('title', ''),
            'summary': entry.get('summary', ''),
            'url': entry.get('link', ''),
            'published': entry.get('published', ''),
            'published_dt': pd.to_datetime(entry.get('published'), errors='coerce')
        })
    df = pd.DataFrame(rows)
    df['source'] = 'World Nuclear News'
    df['data_source'] = 'WNN'
    return df.drop_duplicates(subset=['url']).reset_index(drop=True)

df_wnn = fetch_wnn_rss()
df_wnn.to_parquet('wnn.parquet')
print(f"Saved {len(df_wnn)} WNN articles")

### Listing 11.3 — Google News RSS fetcher with Parquet persistence

In [ ]:
from urllib.parse import quote_plus
import re, html

def strip_html(text):
    if not text: return ""
    text = re.sub(r'<.*?>', ' ', text)
    text = html.unescape(text)
    return re.sub(r'\s+', ' ', text).strip()

def fetch_google_rss(keywords, per_keyword=100):
    rows = []
    for keyword in keywords:
        rss_url = f'https://news.google.com/rss/search?q={quote_plus(keyword)}&hl=en-US&gl=US&ceid=US:en'
        feed = feedparser.parse(rss_url)
        for entry in feed.entries[:per_keyword]:
            title = entry.get('title', '')
            summary_raw = entry.get('summary', '')
            summary_clean = strip_html(summary_raw)
            rows.append({
                'title': title,
                'summary': summary_clean if summary_clean != title else None,
                'url': entry.get('link', ''),
                'source': getattr(getattr(entry, 'source', None), 'title', 'Unknown'),
                'published': entry.get('published', ''),
                'published_dt': pd.to_datetime(entry.get('published'), errors='coerce'),
                'keyword': keyword
            })
    df = pd.DataFrame(rows)
    df['data_source'] = 'GoogleRSS'
    return df.drop_duplicates(subset=['url']).reset_index(drop=True)

df_google = fetch_google_rss(keywords=keywords, per_keyword=100)
df_google.to_parquet('google_rss.parquet')
print(f"Saved {len(df_google)} Google RSS articles")

### Listing 11.4 — GDELT Project fetcher with Parquet persistence

In [ ]:
import requests
import time

def fetch_gdelt(keywords, start_date, end_date, max_per_keyword=250):
    base_url = "http://api.gdeltproject.org/api/v2/doc/doc"

    start_dt = start_date.replace('-', '') + '000000'
    end_dt = end_date.replace('-', '') + '235959'

    rows = []

    for keyword in keywords:
        time.sleep(5)
        params = {
            'query': keyword,
            'mode': 'artlist',
            'maxrecords': max_per_keyword,
            'format': 'json',
            'startdatetime': start_dt,
            'enddatetime': end_dt
        }

        response = requests.get(base_url, params=params, timeout=30)
        try:
            data = response.json()
        except ValueError:
            print(f"GDELT throttled on {keyword}; skipping")
            continue

        for article in data.get('articles', []):
            rows.append({
                'collection_month': start_date[:7],
                'title': article.get('title', ''),
                'url': article.get('url', ''),
                'domain': article.get('domain', ''),
                'language': article.get('language', ''),
                'sourcecountry': article.get('sourcecountry', ''),
                'seendate': article.get('seendate', ''),
                'published_dt': pd.to_datetime(
                    article.get('seendate', '').replace('T', '').replace('Z', ''),
                    format='%Y%m%d%H%M%S',
                    errors='coerce'
                ),
                'keyword': keyword
            })

    df = pd.DataFrame(rows)
    df['data_source'] = 'GDELT'
    if df.empty:  # repo-only guard (not in the book): keep the columns so later cells still run
        print("GDELT returned no articles (all requests throttled?). Wait a few minutes and rerun.")
        return pd.DataFrame(columns=['collection_month', 'title', 'url', 'domain', 'language',
                                     'sourcecountry', 'seendate', 'published_dt', 'keyword', 'data_source'])
    return df.drop_duplicates(subset=['url']).reset_index(drop=True)

# Execute and persist for historical backfill (August-September 2025)
df_gdelt = fetch_gdelt(
    keywords=keywords,
    start_date='2025-08-01',
    end_date='2025-09-30'
)
df_gdelt.to_parquet('gdelt_2025_08_09.parquet')
print(f"Saved {len(df_gdelt)} GDELT articles")

## 11.3 Building the RAG Feature Pipeline
### Listing 11.5 — A toolkit of source-specific processing functions

In [ ]:
import pandas as pd
import numpy as np

# Load all four sources
# NOTE: The printed listing loads timestamped copies from the author's run
# (e.g., newsapi_recent_20251005_0852.parquet). Listings 11.1-11.4 save these
# fixed file names, so we read exactly what the fetchers above wrote.
df_newsapi = pd.read_parquet('newsapi_recent.parquet')
df_wnn = pd.read_parquet('wnn.parquet')
df_google = pd.read_parquet('google_rss.parquet')
df_gdelt = pd.read_parquet('gdelt_2025_08_09.parquet')

print("="*80)
print("STEP 1: DATA CLEANING & PREPARATION")
print("="*80)

# Source-specific processing functions
def process_newsapi(df):
    processed = df[['title', 'description', 'url', 'source_name', 'published_dt']].copy()
    processed.columns = ['title', 'content', 'url', 'source', 'published_dt']
    processed['data_source'] = 'NewsAPI'
    processed['has_content'] = True
    processed['published_dt'] = pd.to_datetime(processed['published_dt']).dt.tz_localize(None)
    return processed

def process_wnn(df):
    processed = df[['title', 'summary', 'url', 'source', 'published_dt']].copy()
    processed.columns = ['title', 'content', 'url', 'source', 'published_dt']
    processed['data_source'] = 'WNN'
    processed['has_content'] = True
    processed['published_dt'] = pd.to_datetime(processed['published_dt']).dt.tz_localize(None)
    return processed

def process_google(df):
    processed = df[['title', 'url', 'source', 'published_dt']].copy()
    # Clean Google RSS titles: remove " - Source" suffix
    processed['title'] = processed['title'].apply(
        lambda x: x.rsplit(' - ', 1)[0] if ' - ' in x else x
    )
    processed['content'] = None
    processed['data_source'] = 'GoogleRSS'
    processed['has_content'] = False
    processed['published_dt'] = pd.to_datetime(processed['published_dt']).dt.tz_localize(None)
    return processed

def process_gdelt(df):
    processed = df[['title', 'url', 'domain', 'published_dt']].copy()
    processed.columns = ['title', 'url', 'source', 'published_dt']
    processed['content'] = None
    processed['data_source'] = 'GDELT'
    processed['has_content'] = False
    processed['published_dt'] = pd.to_datetime(processed['published_dt']).dt.tz_localize(None)
    return processed

### Listing 11.6 — Executing the unification and standardization pipeline

In [ ]:
print("\n[1] Processing each source...")
p_newsapi = process_newsapi(df_newsapi)
p_wnn = process_wnn(df_wnn)
p_google = process_google(df_google)
p_gdelt = process_gdelt(df_gdelt)

print(f"  NewsAPI: {len(p_newsapi)} (content-rich)")
print(f"  WNN: {len(p_wnn)} (content-rich)")
print(f"  Google: {len(p_google)} (title-only, cleaned)")
print(f"  GDELT: {len(p_gdelt)} (title-only)")

# Merge and deduplicate
print("\n[2] Merging and deduplicating...")
df_merged = pd.concat([p_newsapi, p_wnn, p_google, p_gdelt], ignore_index=True)
df_merged = df_merged.sort_values('has_content', ascending=False)
df_merged = df_merged.drop_duplicates(subset=['url'], keep='first')

# Create embedding text
df_merged['text_for_embedding'] = df_merged.apply(
    lambda row: f"{row['title']} {row['content']}"
    if row['has_content'] and pd.notna(row['content'])
    else row['title'],
    axis=1
)

df_merged = df_merged.dropna(subset=['title', 'published_dt'])

# Prepare metadata
df_merged['published_timestamp'] = df_merged['published_dt'].apply(
    lambda x: int(x.timestamp())
)

df_merged['metadata'] = df_merged.apply(
    lambda row: {
        'title': row['title'],
        'source': row['source'],
        'data_source': row['data_source'],
        'published_dt': row['published_dt'].isoformat(),
        'published_ts': row['published_timestamp'],
        'has_content': row['has_content']
    }, axis=1
)

df_merged = df_merged.sort_values('published_dt', ascending=False).reset_index(drop=True)

print(f"  Final dataset: {len(df_merged)} articles")
print(f"  Content-rich: {df_merged['has_content'].sum()}")
print(f"  Title-only: {(~df_merged['has_content']).sum()}")

# Save
df_merged.to_parquet('news_ready_for_embedding.parquet')
print(f"\nSaved: news_ready_for_embedding.parquet")